In [3]:
library(DBI)
library(odbc)
library(dplyr)
library(pROC)
library(car)

Warning message:
"package 'DBI' was built under R version 4.5.3"
Warning message:
"package 'odbc' was built under R version 4.5.3"

Attaching package: 'dplyr'


The following object is masked from 'package:car':

    recode


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Warning message:
"package 'pROC' was built under R version 4.5.3"
Type 'citation("pROC")' for a citation.


Attaching package: 'pROC'


The following objects are masked from 'package:stats':

    cov, smooth, var




In [4]:
con <- dbConnect(odbc::odbc(),
                 Driver = "ODBC Driver 17 for SQL Server",
                 Server = "MSI\\SQLEXPRESS",
                 Database = "Bank",
                 Trusted_Connection = "Yes")

In [5]:
loan_data <- "
SELECT 
    l.account_id,

    CASE 
        WHEN l.loan_quality = 'Bad loan' THEN 1 
        ELSE 0 
    END AS default_flag,

    l.amount,
    l.duration,
    l.monthly_payments,

    d.average_salary,
    d.unemployment_rate_1996,

    SUM(CASE WHEN t.transaction_type = 'Income' THEN t.amount ELSE 0 END) AS total_income,
    SUM(CASE WHEN t.transaction_type IN ('Withdrawal','Spending') THEN t.amount ELSE 0 END) AS total_spending,

    CAST(
        SUM(CASE WHEN t.transaction_type IN ('Withdrawal','Spending') THEN t.amount ELSE 0 END) * 1.0
        /
        NULLIF(SUM(CASE WHEN t.transaction_type = 'Income' THEN t.amount ELSE 0 END), 0)
    AS FLOAT) AS spending_income_ratio

FROM gold.fact_loan l
JOIN gold.fact_transaction t ON l.account_id = t.account_id
JOIN gold.dim_account a ON l.account_id = a.account_id
JOIN gold.dim_district d ON a.district_id = d.district_id

GROUP BY 
    l.account_id,
    l.loan_quality,
    l.amount,
    l.duration,
    l.monthly_payments,
    d.average_salary,
    d.unemployment_rate_1996;
"

df <- dbGetQuery(con, loan_data)


head(df)
summary(df)

,account_id,default_flag,amount,duration,monthly_payments,average_salary,unemployment_rate_1996,total_income,total_spending,spending_income_ratio
,<int>,<int>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,2,0,80952,24,3373,12541,0.43,1597053.5,1554425.8,0.973308
2,19,1,30276,12,2523,9104,2.07,793194.6,782321.3,0.986291
3,25,0,30276,12,2523,9893,4.72,1494372.1,1464173.0,0.979791
4,37,1,318480,60,5308,8547,3.64,497029.2,451124.7,0.907642
5,38,0,110736,48,2307,8402,3.98,308265.3,263684.4,0.855381
6,67,0,165960,24,6915,8427,1.54,2278937.0,2239227.0,0.982575


   account_id     default_flag        amount          duration    
 Min.   :    2   Min.   :0.0000   Min.   :  4980   Min.   :12.00  
 1st Qu.: 2983   1st Qu.:0.0000   1st Qu.: 66534   1st Qu.:24.00  
 Median : 5738   Median :0.0000   Median :116268   Median :36.00  
 Mean   : 5830   Mean   :0.1098   Mean   :150331   Mean   :36.48  
 3rd Qu.: 8686   3rd Qu.:0.0000   3rd Qu.:209148   3rd Qu.:48.00  
 Max.   :11362   Max.   :1.0000   Max.   :590820   Max.   :60.00  
 monthly_payments average_salary  unemployment_rate_1996  total_income    
 Min.   : 304     Min.   : 8110   Min.   :0.430          Min.   :  85796  
 1st Qu.:2451     1st Qu.: 8546   1st Qu.:1.960          1st Qu.: 603976  
 Median :3900     Median : 8992   Median :3.490          Median :1029007  
 Mean   :4165     Mean   : 9519   Mean   :3.487          Mean   :1187179  
 3rd Qu.:5777     3rd Qu.: 9897   3rd Qu.:4.720          3rd Qu.:1577943  
 Max.   :9910     Max.   :12541   Max.   :9.400          Max.   :3724563  
 total

In [6]:
model1 <- glm(default_flag ~ amount + duration + total_income + total_spending + average_salary,
             data = df,
             family = binomial)

summary(model1)


Call:
glm(formula = default_flag ~ amount + duration + total_income + 
    total_spending + average_salary, family = binomial, data = df)

Coefficients:
                 Estimate Std. Error z value Pr(>|z|)    
(Intercept)     5.663e-02  1.089e+00   0.052   0.9585    
amount          6.453e-06  1.481e-06   4.358 1.31e-05 ***
duration       -2.646e-02  1.126e-02  -2.349   0.0188 *  
total_income   -3.626e-05  6.249e-06  -5.803 6.52e-09 ***
total_spending  3.672e-05  6.306e-06   5.823 5.78e-09 ***
average_salary -1.331e-04  1.053e-04  -1.264   0.2061    
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 466.52  on 673  degrees of freedom
Residual deviance: 400.89  on 668  degrees of freedom
AIC: 412.89

Number of Fisher Scoring iterations: 6


**average salary has no meaning because of high P value**

In [7]:
model2 <- glm(default_flag ~ amount + duration + spending_income_ratio,
             data = df,
             family = binomial)

summary(model2)


Call:
glm(formula = default_flag ~ amount + duration + spending_income_ratio, 
    family = binomial, data = df)

Coefficients:
                        Estimate Std. Error z value Pr(>|z|)    
(Intercept)           -3.611e+01  6.075e+00  -5.944 2.78e-09 ***
amount                 5.637e-06  1.607e-06   3.507 0.000453 ***
duration              -2.061e-02  1.169e-02  -1.764 0.077809 .  
spending_income_ratio  3.524e+01  6.246e+00   5.642 1.68e-08 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 466.52  on 673  degrees of freedom
Residual deviance: 394.94  on 670  degrees of freedom
AIC: 402.94

Number of Fisher Scoring iterations: 6


In [ ]:
vif(model2)

amount              duration spending_income_ratio 
             2.294308              2.300509              1.005025

**The VIF values indicate no serious multicollinearity among the predictors.**

**Based on AIC comparison, the model including amount, duration, and spending_income_ratio provides the best fit.**